# Evidentiary range of statistical robustness certification

Reproduction code for the empirical results in *"From Process to Proof"*.

Measures what a sampling-based robustness procedure establishes on
YOLOv8n / COCO person detection under Gaussian pixel noise.

## Experiments

| # | What | Cost |
|---|------|------|
| 1 | Screening: 300 images x 2,000 draws | ~40 min |
| 2 | Calibration: 5 images x 1,000,000 draws | ~6 h |
| 3 | Analysis (no GPU) | seconds |

## Requirements

NVIDIA GPU. Measured on a Tesla T4 at ~255 inferences/s.
CPU execution is ~100x slower and not practical for Experiment 2.

**Reproduction caveat.** Noise is generated on the GPU with `torch.randn`
in single precision. The stream differs from a CPU NumPy stream of the
same nominal seed, so exact replication requires a CUDA device.

## Artifacts produced

- `survey_sigma6_n2000.csv` - per-image screening results
- `cal_set.json` - the pre-registered image selection
- `<image>_sigma6_N1000000.npz` - full score arrays
- `calibration_summary.csv` - per-image calibration ratios
- `manifest.json` - SHA-256 of each artifact

In [ ]:
%pip install -q ultralytics==8.4.60

## 1. Configuration

All parameters are fixed here. The only data-dependent choice in the
protocol is `SIGMA`, selected by the pilot rule documented in section 4.

In [ ]:
from pathlib import Path

# --- detection event ---
IMGSZ          = 640
CONF_THRESHOLD = 0.50     # person score below this = failure
IOU_THRESHOLD  = 0.50     # proposal-to-GT geometric match
MIN_GT_AREA    = 96 * 96  # image eligibility

# --- perturbation ---
SIGMA = 6.0               # 8-bit pixel units; see section 4

# --- experiment 1: screening ---
SURVEY_N       = 300
SURVEY_SAMPLES = 2000
SURVEY_SEED    = 4242     # image selection
CLEAN_SEED     = 0        # unperturbed pass
NOISE_SEED_SV  = 5150     # screening perturbations

# --- experiment 2: calibration ---
CAL_N        = 1_000_000
CAL_SEED_RUN = 20260918   # calibration perturbations
CAL_CHUNK    = 250_000    # progress granularity only
MIN_EVENTS   = 3          # selection rule
TARGET_P     = 1e-3       # selection rule

# --- analysis ---
BOXCOX_SHIFT  = 1e-8
FIT_N         = 1000      # fit-sample size used in prior work
FIT_REPEATS   = 100
ANALYSIS_SEED = 11

# An image measured during earlier exploration, NOT chosen by the
# selection rule. Reported separately wherever it appears.
PILOT_IMAGE = '000000419379.jpg'

ROOT       = Path('/kaggle/working')      # change for other environments
DATA_ROOT  = ROOT / 'coco'
IMAGES_DIR = DATA_ROOT / 'val2017'
OUT        = ROOT / 'calibration'
OUT.mkdir(parents=True, exist_ok=True)

## 2. Data, model, and the measured statistic

For the largest annotated person instance `g` in an image, we track

    s(x) = max { person_score(j) : IoU(box_j, g) >= 0.5 }

over pre-NMS proposals, with failure defined as `s(x) < 0.5`. NMS removes
duplicates and cannot raise a maximum, so it would not change `s`. A trial
with no geometric match yields NaN and counts as a failure.

Noise is applied at native resolution before letterboxing, then clipped
and rounded. Rounding rather than truncation avoids a systematic negative
bias at small sigma.

In [ ]:
import os, json, time, zipfile, urllib.request, hashlib
import numpy as np
import pandas as pd
from scipy import stats
import cv2, torch
import torch.nn.functional as F
from ultralytics import YOLO

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print('WARNING: no GPU. Experiment 2 is not practical here.')

ANN = DATA_ROOT / 'annotations/instances_val2017.json'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)

if not ANN.exists():
    zp = DATA_ROOT / 'ann.zip'
    if not zp.exists():
        urllib.request.urlretrieve(
            'http://images.cocodataset.org/annotations/'
            'annotations_trainval2017.zip', zp)
    with zipfile.ZipFile(zp) as f:
        f.extractall(DATA_ROOT)

coco = json.loads(ANN.read_text())
fname = {im['id']: im['file_name'] for im in coco['images']}
gt_by = {}
for a in coco['annotations']:
    if a['category_id'] != 1 or a.get('iscrowd', 0):
        continue
    x, y, w, h = a['bbox']
    gt_by.setdefault(fname[a['image_id']], []).append([x, y, x + w, y + h])

model = YOLO('yolov8n.pt')
net = model.model.to(DEVICE).float().eval()
assert model.names[0] == 'person' and len(model.names) == 80


def fetch(name):
    p = IMAGES_DIR / name
    if not p.exists():
        urllib.request.urlretrieve(
            'http://images.cocodataset.org/val2017/' + name, p)
    im = cv2.imread(str(p))
    if im is None:
        raise RuntimeError('imread failed: ' + name)
    return im


def target_box(name):
    return max(gt_by[name], key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))


def iou_gpu(boxes, gt):
    lt = torch.maximum(boxes[..., :2], gt[:2])
    rb = torch.minimum(boxes[..., 2:], gt[2:])
    inter = (rb - lt).clamp(min=0).prod(-1)
    area = (boxes[..., 2:] - boxes[..., :2]).clamp(min=0).prod(-1)
    union = area + (gt[2:] - gt[:2]).prod() - inter
    return inter / union.clamp(min=1e-12)


@torch.inference_mode()
def matched_scores(img_bgr, gt_orig, sigma, n, gen, batch=128):
    """Return n draws of s(x). NaN where no proposal matched the GT."""
    h, w = img_bgr.shape[:2]
    ratio = min(IMGSZ / h, IMGSZ / w)
    nw, nh = round(w * ratio), round(h * ratio)
    left = round((IMGSZ - nw) / 2 - 0.1)
    top = round((IMGSZ - nh) / 2 - 0.1)
    g = np.asarray(gt_orig, dtype=np.float32)
    gt_lb = torch.tensor([g[0] * ratio + left, g[1] * ratio + top,
                          g[2] * ratio + left, g[3] * ratio + top],
                         device=DEVICE)
    base = torch.from_numpy(np.ascontiguousarray(
        img_bgr[:, :, ::-1].transpose(2, 0, 1))).to(DEVICE).float()

    out, done = [], 0
    while done < n:
        b = min(batch, n - done)
        x0 = base.unsqueeze(0).expand(b, -1, -1, -1).contiguous()
        if sigma > 0:
            noise = torch.randn(x0.shape, device=DEVICE, generator=gen)
            x0 = torch.clamp(torch.round(x0 + noise * sigma), 0, 255)
        r = F.interpolate(x0, size=(nh, nw), mode='bilinear',
                          align_corners=False)
        x = torch.full((b, 3, IMGSZ, IMGSZ), 114.0, device=DEVICE)
        x[:, :, top:top + nh, left:left + nw] = r

        d = net(x / 255.0)
        d = d[0] if isinstance(d, (tuple, list)) else d
        if d.ndim != 3 or d.shape[1] != 84:
            raise RuntimeError('unexpected head layout ' + str(tuple(d.shape)))

        xywh = d[:, :4, :].permute(0, 2, 1)
        boxes = torch.cat([xywh[..., :2] - xywh[..., 2:] / 2,
                           xywh[..., :2] + xywh[..., 2:] / 2], -1)
        person = d[:, 4, :]
        s = torch.where(iou_gpu(boxes, gt_lb) >= IOU_THRESHOLD,
                        person, torch.full_like(person, -1.0)).max(1).values
        out.append(torch.where(s < 0, torch.full_like(s, float('nan')),
                               s).cpu())
        done += b
    return torch.cat(out).numpy()


print('device:', DEVICE, '| person images:', len(gt_by))

## 3. Experiment 1: screening

300 images drawn uniformly from the eligible pool, 2,000 perturbations
each at fixed sigma. Images failing detection on the unperturbed frame
are recorded with `clean_detected=False` rather than dropped.

`Z = (mean(s) - 0.5) / sd(s)` is a scale-free distance to the decision
threshold. It is **not** converted to a probability anywhere.

The perturbation stream advances continuously across images, so a resumed
run does not reproduce the same draws. Acceptable for screening; not for
calibration.

Runtime: ~40 min.

In [ ]:
SURVEY_CSV = OUT / ('survey_sigma%g_n%d.csv' % (SIGMA, SURVEY_SAMPLES))

if SURVEY_CSV.exists():
    survey = pd.read_csv(SURVEY_CSV)
    print('loaded existing survey:', len(survey), 'rows')
else:
    pool = [n for n in sorted(gt_by)
            if max((b[2] - b[0]) * (b[3] - b[1])
                   for b in gt_by[n]) >= MIN_GT_AREA]
    idx = np.random.default_rng(SURVEY_SEED).choice(
        len(pool), min(SURVEY_N, len(pool)), replace=False)
    names = [pool[i] for i in idx]
    print('pool=%d  screening=%d  sigma=%g' % (len(pool), len(names), SIGMA))

    gclean = torch.Generator(device=DEVICE).manual_seed(CLEAN_SEED)
    gnoise = torch.Generator(device=DEVICE).manual_seed(NOISE_SEED_SV)
    rows, t0 = [], time.perf_counter()

    for i, nm in enumerate(names, 1):
        im = fetch(nm)
        g = target_box(nm)
        cs = float(matched_scores(im, g, 0.0, 1, gclean)[0])
        area = float((g[2] - g[0]) * (g[3] - g[1]))

        if not np.isfinite(cs) or cs < CONF_THRESHOLD:
            rows.append(dict(image=nm, clean_score=cs, clean_detected=False,
                             mean=np.nan, sd=np.nan, min=np.nan,
                             z_to_thr=np.nan, p_fail=np.nan,
                             coverage=np.nan, gt_area=area))
        else:
            v = matched_scores(im, g, SIGMA, SURVEY_SAMPLES, gnoise)
            f = v[np.isfinite(v)]
            z = np.nan_to_num(v, nan=0.0)
            sd = float(f.std(ddof=1))
            rows.append(dict(
                image=nm, clean_score=cs, clean_detected=True,
                mean=float(f.mean()), sd=sd, min=float(f.min()),
                z_to_thr=float((f.mean() - CONF_THRESHOLD) / sd),
                p_fail=float((z < CONF_THRESHOLD).mean()),
                coverage=float(np.isfinite(v).mean()), gt_area=area))

        if i % 25 == 0 or i == len(names):
            pd.DataFrame(rows).to_csv(SURVEY_CSV, index=False)
            el = time.perf_counter() - t0
            print('%4d/%d  %5.1f min  eta %5.1f min'
                  % (i, len(names), el / 60,
                     (el / i * len(names) - el) / 60), flush=True)

    survey = pd.DataFrame(rows)
    survey.to_csv(SURVEY_CSV, index=False)

d = survey[survey.clean_detected]
print('')
print('%d screened, %d clean-detected (%.1f%%)'
      % (len(survey), len(d), 100 * len(d) / len(survey)))
print('with observed failures : %d/%d' % ((d.p_fail > 0).sum(), len(d)))
print('sd across images       : [%.4f, %.4f] = %.0fx'
      % (d.sd.min(), d.sd.max(), d.sd.max() / d.sd.min()))
print('Z to threshold         : median %.1f, range [%.1f, %.1f]'
      % (d.z_to_thr.median(), d.z_to_thr.min(), d.z_to_thr.max()))

pf = np.sort(d.p_fail.values)[::-1]
tot = pf.sum()
for frac in (0.05, 0.10):
    k = max(1, int(round(frac * len(pf))))
    print('top %3.0f%% of images carry %5.1f%% of observed failures'
          % (100 * frac, 100 * pf[:k].sum() / tot))

## 4. Selection rule (pre-registered)

The five calibration images are chosen by a rule fixed before the
calibration runs:

> the five images whose screened failure rate is closest to 1e-3 on a log
> scale, among those with at least 3 observed failures in the 2,000-draw
> screen.

The minimum-event threshold matters: a rate estimated from a single event
carries a confidence interval spanning roughly two orders of magnitude, so
selecting on proximity alone would select noise.

### Choice of sigma

`SIGMA = 6` was selected on one image by a rule declared in advance: the
smallest sigma whose observed failure rate over 2,000 draws fell in
[1e-3, 1e-2]. The pilot gave 0, 2.0e-3, 1.5e-2, 7.2e-2 for
sigma = 5, 6, 7, 8. All pilot draws were discarded.

The transition is sharp, and results at this sigma should not be read as
characterising other perturbation magnitudes.

In [ ]:
CAL_JSON = OUT / 'cal_set.json'
RULE = ('5 images whose screened failure rate is closest to %.0e on a log '
        'scale, among those with at least %d observed failures in the '
        '%d-draw screen' % (TARGET_P, MIN_EVENTS, SURVEY_SAMPLES))

if CAL_JSON.exists():
    CAL_SET = json.loads(CAL_JSON.read_text())['images']
    print('loaded existing selection')
else:
    c = d[(d.p_fail * SURVEY_SAMPLES).round() >= MIN_EVENTS].copy()
    c['dist'] = np.abs(np.log10(c.p_fail) - np.log10(TARGET_P))
    sel = c.nsmallest(5, 'dist').sort_values('p_fail')
    CAL_SET = sel.image.tolist()
    CAL_JSON.write_text(json.dumps(dict(
        rule=RULE, min_events=MIN_EVENTS, target_p=TARGET_P,
        candidates=int(len(c)), images=CAL_SET,
        screened={r.image: dict(
            p_fail=float(r.p_fail), clean=float(r.clean_score),
            events=int(round(r.p_fail * SURVEY_SAMPLES)))
            for _, r in sel.iterrows()}), indent=2))
    print('%d candidates met the event threshold' % len(c))

for nm in CAL_SET:
    r = survey[survey.image == nm].iloc[0]
    print('%13s  clean=%.4f  p_fail=%.1e  events=%3d  sd=%.4f'
          % (nm[:-4], r.clean_score, r.p_fail,
             int(round(r.p_fail * SURVEY_SAMPLES)), r.sd))

## 5. Experiment 2: calibration

1,000,000 draws per selected image. Order statistics of this sample serve
as the reference against which the parametric estimate is compared.

Each image uses the same seed with a fresh generator, so the stream is
reproducible per image and independent of chunking.

Runtime: ~73 min per image, ~6 h total.

In [ ]:
import gc

for qi, nm in enumerate(CAL_SET, 1):
    p = OUT / ('%s_sigma%g_N%d.npz' % (nm[:-4], SIGMA, CAL_N))
    tag = '[%d/%d] %s' % (qi, len(CAL_SET), nm)
    if p.exists():
        print(tag + ': already done')
        continue

    im, g = fetch(nm), target_box(nm)
    gr = torch.Generator(device=DEVICE).manual_seed(CAL_SEED_RUN)
    t0, parts = time.perf_counter(), []
    for done in range(0, CAL_N, CAL_CHUNK):
        n = min(CAL_CHUNK, CAL_N - done)
        parts.append(matched_scores(im, g, SIGMA, n, gr))
        el = time.perf_counter() - t0
        fr = (done + n) / CAL_N
        print('%s  %9d  %5.1f min  eta %5.1f min'
              % (tag, done + n, el / 60, (el / fr - el) / 60), flush=True)
    sc = np.concatenate(parts).astype(np.float32)
    del parts

    meta = dict(image=nm, sigma=float(SIGMA), n=int(CAL_N),
                seed=int(CAL_SEED_RUN), gt=list(map(float, g)),
                conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, imgsz=IMGSZ,
                model='yolov8n.pt', selection_rule=RULE,
                torch=torch.__version__,
                device=(torch.cuda.get_device_name(0)
                        if DEVICE != 'cpu' else 'cpu'))
    tmp = p.with_suffix('.tmp.npz')
    np.savez_compressed(tmp, scores=sc, meta=json.dumps(meta))
    tmp.replace(p)

    z = np.nan_to_num(sc, nan=0.0)
    print('%s  saved  p_fail=%.3e  events=%d'
          % (tag, (z < CONF_THRESHOLD).mean(), (z < CONF_THRESHOLD).sum()),
          flush=True)
    print('')
    del sc, z, im
    gc.collect()
    if DEVICE != 'cpu':
        torch.cuda.empty_cache()

print('calibration complete')

## 6. Analysis

No GPU required. Reproduces the tables in the paper from the stored score
arrays.

The fit samples are drawn from the same pool as the reference quantiles,
so the two are not independent. An independent reference would be
preferable; we expect it to widen rather than narrow the reported
discrepancies.

In [ ]:
TARGETS = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]
KINDS = ['raw', 'boxcox', 'yeojohnson', 'log', 'logit']


def fit_transform(sample, kind='boxcox'):
    """Return (cdf_callable, lambda_or_None, ad_non_rejected)."""
    if kind == 'raw':
        tr, lam, g = sample, None, (lambda x: x)
    elif kind == 'boxcox':
        tr, lam = stats.boxcox(sample + BOXCOX_SHIFT)
        g = (lambda x, l=lam: np.log(x + BOXCOX_SHIFT) if l == 0
             else ((x + BOXCOX_SHIFT) ** l - 1) / l)
    elif kind == 'yeojohnson':
        tr, lam = stats.yeojohnson(sample)
        g = (lambda x, l=lam:
             float(stats.yeojohnson(np.array([x]), lmbda=l)[0]))
    elif kind == 'log':
        tr, lam = np.log(sample + BOXCOX_SHIFT), None
        g = lambda x: np.log(x + BOXCOX_SHIFT)
    else:
        e = np.clip(sample, 1e-6, 1 - 1e-6)
        tr, lam = np.log(e / (1 - e)), None
        c = lambda x: np.clip(x, 1e-6, 1 - 1e-6)
        g = lambda x: np.log(c(x) / (1 - c(x)))
    m, sd = tr.mean(), tr.std(ddof=1)
    ad = stats.anderson(tr, dist='norm')
    ok = bool(ad.statistic < ad.critical_values[2])   # 5% level
    return (lambda x: float(stats.norm.cdf((g(x) - m) / sd))), lam, ok


def load(path):
    z = np.nan_to_num(np.load(path)['scores'], nan=0.0).astype(np.float64)
    return z, z[z > 0], np.sort(z)


files = sorted(OUT.glob('*_sigma%g_N%d.npz' % (SIGMA, CAL_N)))
print('%d calibration files' % len(files))
print('')

summary = []
for path in files:
    z, fin, order = load(path)
    N = len(z)
    measured = (z < CONF_THRESHOLD).mean()
    rng = np.random.default_rng(ANALYSIS_SEED)
    subs = [rng.choice(fin, FIT_N, replace=False) for _ in range(FIT_REPEATS)]
    fits = [fit_transform(s, 'boxcox') for s in subs]
    lams = np.array([l for _, l, _ in fits])
    adp = sum(o for _, _, o in fits)

    name = path.name[:12]
    flag = ('  [exploratory, not selected by rule]'
            if name + '.jpg' == PILOT_IMAGE else '')
    print('=== ' + name + flag)
    print('    measured p_fail = %.3e   mean=%.4f  sd=%.4f'
          % (measured, fin.mean(), fin.std(ddof=1)))
    print('    lambda = %6.2f +/- %.2f   AD non-rejected %d/%d'
          % (lams.mean(), lams.std(ddof=1), adp, FIT_REPEATS))
    print('     measured p    score    estimate     ratio')
    row = dict(image=name, mean=fin.mean(), sd=fin.std(ddof=1),
               lam=lams.mean(), ad=adp, measured=measured,
               selected=(name + '.jpg' != PILOT_IMAGE))
    for t in TARGETS:
        q = order[int(round(t * N))]
        est = np.median([f(q) for f, _, _ in fits])
        print('    %11.0e %8.4f %11.3e %9.1f' % (t, q, est, est / t))
        row['r%.0e' % t] = est / t
    est_thr = np.median([f(CONF_THRESHOLD) for f, _, _ in fits])
    row['r_thr'] = est_thr / measured
    print('    at threshold 0.5: estimate=%.3e  ratio=%.2f'
          % (est_thr, est_thr / measured))
    print('')
    summary.append(row)

S = pd.DataFrame(summary).sort_values('sd')
r5 = S['r1e-05']
print('ratio at 1e-5 spans %.0fx [%.2f, %.2f]'
      % (r5.max() / r5.min(), r5.min(), r5.max()))
print('inputs underestimating the measured quantile: %d/%d'
      % ((r5 < 1).sum(), len(S)))
print('')
print('Spearman lambda vs sd            : rho=%+.3f'
      % stats.spearmanr(S.lam, S.sd)[0])
print('Spearman log10(ratio) vs lambda  : rho=%+.3f'
      % stats.spearmanr(np.log10(r5), S.lam)[0])
S.to_csv(OUT / 'calibration_summary.csv', index=False)

### 6.1 Transform comparison on the worst case

The input where the parametric estimate falls **below** the measured
quantile is the one that matters most for a safety argument: there the
method understates risk.

In [ ]:
worst = S.loc[S['r1e-05'].idxmin(), 'image']
path = OUT / ('%s_sigma%g_N%d.npz' % (worst, SIGMA, CAL_N))
z, fin, order = load(path)
measured = (z < CONF_THRESHOLD).mean()
rng = np.random.default_rng(ANALYSIS_SEED)
subs = [rng.choice(fin, FIT_N, replace=False) for _ in range(FIT_REPEATS)]

print('%s: lowest estimate-to-measured ratio' % worst)
print('measured p_fail at %.2f = %.3e' % (CONF_THRESHOLD, measured))
print('')
print('   transform    est@1e-5   ratio     est@0.5   ratio        AD')
for k in KINDS:
    built = []
    for s in subs:
        try:
            built.append(fit_transform(s, k))
        except Exception:
            pass
    if not built:
        print('%12s      failed' % k)
        continue
    e5 = np.median([f(order[10]) for f, _, _ in built])
    et = np.median([f(CONF_THRESHOLD) for f, _, _ in built])
    ok = sum(o for _, _, o in built)
    print('%12s %11.2e %7.2f %11.2e %7.2f %5d/%d'
          % (k, e5, e5 / 1e-5, et, et / measured, ok, len(built)))

## 7. Manifest

SHA-256 of every artifact, so the figures in the paper can be traced to
the files that produced them.

In [ ]:
man = []
for p in sorted(OUT.glob('*')):
    if p.is_file() and p.suffix != '.tmp':
        man.append(dict(file=p.name, bytes=p.stat().st_size,
                        sha256=hashlib.sha256(p.read_bytes()).hexdigest()))
(OUT / 'manifest.json').write_text(json.dumps(man, indent=2))
for m in man:
    print('%-45s %8.2f MB  %s'
          % (m['file'], m['bytes'] / 1e6, m['sha256'][:16]))